# Session 7.2: Advanced CNNs - CIFAR-10 Classification

**Learning Objectives:**
- Build deeper CNN architectures
- Work with color images (RGB)
- Apply data augmentation techniques
- Achieve >70% accuracy on CIFAR-10
- Understand CNN design patterns

**Time Estimate:** 90-120 minutes

---

## 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, classification_report

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")

## 2. Load CIFAR-10 Dataset

CIFAR-10:
- 60,000 32x32 color images
- 10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck
- More challenging than MNIST!

In [ ]:
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Training shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Image shape: {X_train[0].shape}")
print(f"Classes: {class_names}")

### Visualization 1: Sample CIFAR-10 Images

In [ ]:
fig, axes = plt.subplots(5, 10, figsize=(15, 8))
fig.suptitle('Sample CIFAR-10 Images (50 samples)', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i])
    ax.set_title(class_names[y_train[i][0]], fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

### Visualization 2: Class Distribution

In [ ]:
unique, counts = np.unique(y_train, return_counts=True)

plt.figure(figsize=(12, 6))
plt.bar(range(10), counts, color='steelblue', edgecolor='black')
plt.title('CIFAR-10 Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Class', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(range(10), class_names, rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Normalize to [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# One-hot encode labels
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

print(f"Normalized data range: [{X_train.min()}, {X_train.max()}]")
print(f"Label shape: {y_train_cat.shape}")

## 4. Data Augmentation

Data augmentation helps prevent overfitting by artificially expanding the training set.

In [ ]:
# Create data generator with augmentation
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

datagen.fit(X_train)
print("Data augmentation configured successfully.")

### Visualization 3: Augmented Images

In [ ]:
# Show original and augmented versions
sample_img = X_train[0].reshape((1, 32, 32, 3))

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Original vs Augmented Images', fontsize=14, fontweight='bold')

# Original
axes[0, 0].imshow(X_train[0])
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

# Augmented versions
count = 0
for batch in datagen.flow(sample_img, batch_size=1):
    if count < 9:
        row = (count + 1) // 5
        col = (count + 1) % 5
        axes[row, col].imshow(batch[0])
        axes[row, col].set_title(f'Augmented {count+1}')
        axes[row, col].axis('off')
        count += 1
    else:
        break

plt.tight_layout()
plt.show()

## 5. Build Advanced CNN Architecture

**Architecture Design Pattern:**
- Increasing depth: 32 → 64 → 128 filters
- Decreasing spatial size through pooling
- Batch normalization for faster training
- Dropout for regularization

### LLM Prompt:
```
Create a deep CNN for CIFAR-10 with:
- 3 convolutional blocks, each with 2 Conv2D layers + BatchNorm + MaxPooling
- Filter progression: 32, 64, 128
- Dense layers: 256 units with dropout
- Output: 10 classes with softmax
```

In [ ]:
def create_advanced_cnn():
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),
        
        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.4),
        
        # Dense layers
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    
    return model

model = create_advanced_cnn()
model.summary()

## 6. Compile and Train

In [ ]:
# Compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

print("Model compiled with callbacks configured.")

In [ ]:
# Train with data augmentation
print("Training advanced CNN...\n")

history = model.fit(
    datagen.flow(X_train, y_train_cat, batch_size=64),
    steps_per_epoch=len(X_train) // 64,
    epochs=50,
    validation_data=(X_test, y_test_cat),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print("\nTraining completed!")

### Visualization 4: Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train Acc', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Val Acc', linewidth=2)
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], label='Train Loss', linewidth=2)
ax2.plot(history.history['val_loss'], label='Val Loss', linewidth=2)
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Evaluate on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

if test_acc > 0.70:
    print("\n✓ SUCCESS: Achieved >70% accuracy target!")
else:
    print(f"\n⚠ Current accuracy: {test_acc:.2%}. Target: >70%")

### Visualization 5: Confusion Matrix

In [ ]:
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = y_test.flatten()

cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - CIFAR-10', fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred_classes, target_names=class_names))

### Visualization 6: Per-Class Accuracy

In [ ]:
# Calculate per-class accuracy
class_acc = cm.diagonal() / cm.sum(axis=1)

plt.figure(figsize=(12, 6))
bars = plt.bar(range(10), class_acc, color='steelblue', edgecolor='black')
plt.title('Per-Class Accuracy', fontsize=14, fontweight='bold')
plt.xlabel('Class')
plt.ylabel('Accuracy')
plt.xticks(range(10), class_names, rotation=45)
plt.axhline(y=test_acc, color='r', linestyle='--', label=f'Overall: {test_acc:.2%}')
plt.ylim([0, 1])
plt.grid(axis='y', alpha=0.3)
plt.legend()

for i, bar in enumerate(bars):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{class_acc[i]:.1%}',
             ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# Identify hardest classes
worst_idx = np.argmin(class_acc)
best_idx = np.argmax(class_acc)
print(f"\nHardest class: {class_names[worst_idx]} ({class_acc[worst_idx]:.2%})")
print(f"Easiest class: {class_names[best_idx]} ({class_acc[best_idx]:.2%})")

### Visualization 7: Sample Predictions

In [ ]:
# Display predictions
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle('Sample Predictions', fontsize=16, fontweight='bold')

indices = np.random.choice(len(X_test), 32, replace=False)

for i, (ax, idx) in enumerate(zip(axes.flat, indices)):
    ax.imshow(X_test[idx])
    pred = y_pred_classes[idx]
    true = y_true[idx]
    conf = y_pred[idx][pred]
    
    color = 'green' if pred == true else 'red'
    ax.set_title(f'T:{class_names[true]}\nP:{class_names[pred]}\n{conf:.1%}',
                 fontsize=7, color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 8. Save Model

In [ ]:
model.save('cifar10_cnn_model.h5')
print("Model saved successfully.")

## 9. Key Takeaways

### Architecture Improvements:
- **Deeper networks**: More conv layers capture complex features
- **Batch normalization**: Faster training, better convergence
- **Data augmentation**: Reduces overfitting significantly
- **Progressive dropout**: Stronger regularization in later layers

### CIFAR-10 Insights:
- Much harder than MNIST (70-80% vs 98%)
- Color images (3 channels) require more parameters
- Similar classes (cat/dog) are harder to distinguish
- Data augmentation is essential

### Next Session:
Session 7.3: Transfer Learning - Use pre-trained models to achieve >90% accuracy!

---

**Session Complete!**